# DTS304TC Assessment 2 — Part 3 (**Individual Notebook**)

**This notebook is for the individual submission.**  
Each student must submit **their own copy** of this notebook for **Part C / Part D**.

This revised version makes the split usage explicit:
- **train**: training / ablations
- **val**: all development, small-object analysis, comparison, model selection
- **test**: one final hold-out confirmation only for the chosen final method


## Author
- Yuyao Wang (individual part of a group project)
- Assigned focus classes are derived from the last digit of the student ID (3): Car_Bus and Pedestrian

---


In [ ]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import data_aug as aug
from dataset import camvidLoader, from_label_to_rgb
from eval_obj_metric import compute_object_level_stats_percentile
from unet import UNet


In [ ]:
# ----------------------------
# Basic configuration
# ----------------------------
SEED = 42
IMG_SIZE = [384, 384]
BATCH_SIZE = 32
NUM_WORKERS = 0
MODEL_PATH = "camvid_sunny_model.pt"
DATA_ROOT = "../CamVid/sunny"

CLASS_LABELS_DICT = {
    "Sky": 0,
    "Building": 1,
    "Pole": 2,
    "Road": 3,
    "LaneMarking": 4,
    "SideWalk": 5,
    "Pavement": 6,
    "Tree": 7,
    "SignSymbol": 8,
    "Fence": 9,
    "Car_Bus": 10,
    "Pedestrian": 11,
    "Bicyclist": 12,
    "Unlabelled": 13,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_LABELS_DICT.items()}
NUM_CLASSES = len(CLASS_LABELS_DICT)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Model path exists:", Path(MODEL_PATH).exists())
print("Data root exists:", Path(DATA_ROOT).exists())

In [ ]:
def assigned_focus_classes(last_digit):
    last_digit = int(last_digit)
    if last_digit in {0, 3, 6, 9}:
        return ("Car_Bus", "Pedestrian")
    if last_digit in {1, 4, 7}:
        return ("Car_Bus", "LaneMarking")
    if last_digit in {2, 5, 8}:
        return ("Pedestrian", "LaneMarking")
    raise ValueError("Student ID last digit must be in 0-9.")


# Fill this in before running the notebook:
STUDENT_ID_LAST_DIGIT = 3

FOCUS_CLASSES = assigned_focus_classes(STUDENT_ID_LAST_DIGIT)
print("Assigned focus classes:", FOCUS_CLASSES)

In [ ]:
def load_trained_model(model_path, device, num_classes=14):
    try:
        obj = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        obj = torch.load(model_path, map_location=device)

    if isinstance(obj, torch.nn.Module):
        model = obj
    elif isinstance(obj, dict):
        model = UNet(3, num_classes, width=32, bilinear=True)
        model.load_state_dict(obj)
    else:
        raise TypeError(f"Unsupported checkpoint type: {type(obj)}")

    model = model.to(device)
    return model

In [ ]:
# Evaluation helper functions (reused across C1 and C3)

def global_accuracy_metric(y_true, y_pred):
    """Fraction of correctly classified pixels."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.sum(y_true == y_pred) / y_true.size


def iou_per_class(y_true, y_pred, num_classes=14):
    """Per-class IoU for a single image; NaN when class absent from both GT and pred."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    ious = []
    for c in range(num_classes):
        tp = np.logical_and(y_true == c, y_pred == c).sum()
        fp = np.logical_and(y_true != c, y_pred == c).sum()
        fn = np.logical_and(y_true == c, y_pred != c).sum()
        denom = tp + fp + fn
        ious.append(np.nan if denom == 0 else tp / denom)
    return np.array(ious, dtype=np.float64)


def evaluate_model_on_loader(model, loader, device, num_classes=14):
    """Run inference on a DataLoader; return per-image and dataset-level metrics."""
    model.eval()
    image_accuracies, image_ious, image_paths = [], [], []
    gt_masks, pred_masks = [], []

    with torch.no_grad():
        for images, labels, paths in loader:
            images = images.to(device)
            preds = torch.argmax(model(images), dim=1).cpu().numpy()
            labels_np = labels.numpy()
            for y_true, y_pred, p in zip(labels_np, preds, paths):
                image_accuracies.append(global_accuracy_metric(y_true, y_pred))
                image_ious.append(iou_per_class(y_true, y_pred, num_classes))
                image_paths.append(p)
                gt_masks.append(y_true)
                pred_masks.append(y_pred)

    image_ious_arr = np.array(image_ious, dtype=np.float64)
    per_class_iou = np.nanmean(image_ious_arr, axis=0)
    return {
        "overall_global_accuracy": np.nanmean(image_accuracies),
        "per_class_iou": per_class_iou,
        "mean_iou": float(np.nanmean(per_class_iou)),
        "image_accuracies": np.array(image_accuracies),
        "image_ious": image_ious_arr,
        "image_paths": image_paths,
        "gt_masks": gt_masks,
        "pred_masks": pred_masks,
    }


# Val DataLoader (sunny) + baseline inference

val_dataset = camvidLoader(
    root=DATA_ROOT,
    split="val",
    is_aug=False,
    img_size=IMG_SIZE,
    is_pytorch_transform=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)
print("Val images:", len(val_dataset))

# Load pretrained model in eval mode
baseline_model = load_trained_model(MODEL_PATH, device, NUM_CLASSES)
baseline_model.eval()

# Run baseline inference: val split only, no test data used here
val_results = evaluate_model_on_loader(baseline_model, val_loader, device, NUM_CLASSES)
gt_val_baseline = np.array(val_results["gt_masks"])    # shape (92, 384, 384)
pred_val_baseline = np.array(val_results["pred_masks"])  # shape (92, 384, 384)

print(f"\n=== Baseline (Pretrained) | Sunny Val ===")
print(f"Global Accuracy : {val_results['overall_global_accuracy']:.4f}")
print(f"Mean IoU        : {val_results['mean_iou']:.4f}")
print("\nPer-Class IoU:")
for idx, iou in enumerate(val_results["per_class_iou"]):
    name = IDX_TO_CLASS[idx]
    print(f"  {idx:02d} {name:<14}: {'nan' if np.isnan(iou) else f'{iou:.4f}'}")


## Part C1 — baseline small-object analysis on the **sunny validation split**

Use the pretrained baseline first.  
Your priority is the **two assigned focus classes**, not all three classes at once.


In [ ]:
# # Run object-level analysis for your assigned classes only.
# # You may add a related extra class if it helps your discussion, but the assigned pair is the priority.

# # Task C1: Please read and understand our evaluation code function compute_object_level_stats_percentile
# # and use Run compute_object_level_stats_percentile for car, lane-mark, and pedestrian classes.
# # Compare small-object metrics with large-object metrics to validate that segmentation of small objects is indeed a challenge for the car, lane-marker, and pedestrian classes.
# # Show at least five images with ground-truth segmentations vs. model prediction results highlighting missed or partially segmented small objects.

# # Example: 
# # gt_mask_list is an numpy array of shape (92, 384, 384), indicating 140 groundtruth segmentation mask of 384x384 (data type is int64)
# # pred_mask_list is an numpy array of shape (92, 384, 384), indicating 140 prediction segmentation mask of 384x384 (data type is int64)
# # print(compute_object_level_stats_percentile(np.array(gt_mask),np.array(pred_mask), 'LaneMarking', iou_threshold=0.25, size_percentile=50)) # lanemarking
# # wil output
# #Metric                        Small Objects       Large Objects       
# #======================================================================
# #Class Name                    LaneMarking         
# #Size Threshold (pixels)       4.00                
# #======================================================================
# #Total Ground Truths           3620                3904                
# #Total Predicted               3051                3089                
# #Precision                     0.0960              0.4618              
# #Recall                        0.0796              0.3809              
# #F1 Score                      0.0870              0.4175              
# #False Positives (FP)          2711                1733                
# #False Negatives (FN)          3332                2417                
# #True Positives (TP)           288                 1487                
# #======================================================================
# #(0.0960320106702202, 0.07955801104972156, 0.08702220874330649, 0.4618012422360105, 0.3808913934426132, 0.4174620998943009)
# # Please include this output in your notebook and copy this table to your Assessment Task 2 Worksheet as well (this table will not be counted towards word limit or page limit)
# # Please do the evaluation similar for other class like:
# # compute_object_level_stats_percentile(np.array(gt_mask),np.array(pred_mask), 'Car_Bus', iou_threshold=0.2, size_percentile=50)
# # compute_object_level_stats_percentile(np.array(gt_mask),np.array(pred_mask), 'Pedestrian', iou_threshold=0.2, size_percentile=50)

# '''
# baseline_object_results = {}
# for class_name in FOCUS_CLASSES:
#     print("\n" + "=" * 90)
#     print("Baseline object-level results for:", class_name)
#     baseline_object_results[class_name] = compute_object_level_stats_percentile(
#         gt_masks=gt_val_baseline,
#         pred_masks=pred_val_baseline,
#         class_label=class_name,
#         iou_threshold=0.2,
#         size_percentile=50,
#     )
# '''


In [ ]:
# Small-object vs large-object metrics (Car_Bus + Pedestrian)
# Focus classes assigned by student ID last digit 3 -> Car_Bus + Pedestrian

FOCUS_CLASS_IDS = {cls: CLASS_LABELS_DICT[cls] for cls in FOCUS_CLASSES}
print("Focus classes:", FOCUS_CLASSES)

baseline_object_results = {}
for class_name in FOCUS_CLASSES:
    print("\n" + "=" * 70)
    print(f"Baseline object-level results | {class_name}")
    baseline_object_results[class_name] = compute_object_level_stats_percentile(
        gt_masks=gt_val_baseline,
        pred_masks=pred_val_baseline,
        class_label=class_name,
        iou_threshold=0.2,
        size_percentile=50,
    )


In [ ]:
# Visual examples, at least 5 failure cases per focus class

def find_worst_class_images(gt_masks, pred_masks, class_id, top_n=5, min_gt_pixels=30):
    """
    Return indices of images where the class is present in GT
    but the model performs worst (lowest per-class IoU).
    min_gt_pixels filters out images where the class is nearly absent.
    """
    scores = []
    for i, (gt, pred) in enumerate(zip(gt_masks, pred_masks)):
        gt_cls = gt == class_id
        if gt_cls.sum() < min_gt_pixels:
            continue
        pred_cls = pred == class_id
        tp = np.logical_and(gt_cls, pred_cls).sum()
        fp = np.logical_and(~gt_cls, pred_cls).sum()
        fn = np.logical_and(gt_cls, ~pred_cls).sum()
        denom = tp + fp + fn
        iou = tp / denom if denom > 0 else 0.0
        scores.append((i, iou, int(gt_cls.sum())))
    scores.sort(key=lambda x: x[1])  # ascending = worst first
    return scores[:top_n]


def show_failure_examples(scores, gt_masks, pred_masks, image_paths, class_name, class_id):
    """Side-by-side: original image | GT mask | Prediction mask."""
    n = len(scores)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row, (img_idx, iou, n_gt_px) in enumerate(scores):
        gt = gt_masks[img_idx]
        pred = pred_masks[img_idx]
        img = plt.imread(image_paths[img_idx])

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(
            f"Image {img_idx} | {class_name} GT pixels: {n_gt_px}\n{class_name} IoU = {iou:.3f}",
            fontsize=9,
        )
        axes[row, 0].axis("off")

        axes[row, 1].imshow(from_label_to_rgb(gt))
        axes[row, 1].set_title("Ground Truth", fontsize=9)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(from_label_to_rgb(pred))
        axes[row, 2].set_title("Prediction", fontsize=9)
        axes[row, 2].axis("off")

    fig.suptitle(
        f"C1 | Worst {class_name} Segmentation Cases (Baseline, Val Split)",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


# Show 5 worst cases for each assigned focus class
for cls_name, cls_id in FOCUS_CLASS_IDS.items():
    worst = find_worst_class_images(
        gt_val_baseline, pred_val_baseline,
        class_id=cls_id,
        top_n=5,
        min_gt_pixels=30,
    )
    show_failure_examples(
        worst, gt_val_baseline, pred_val_baseline,
        val_results["image_paths"],
        class_name=cls_name,
        class_id=cls_id,
    )


## Part C2 — improvement design and ablations

Use the training scaffold below to test your own method.  
You still need to justify:
- what you changed
- why it should help your assigned small-object problem
- what trade-off you expected
- what each ablation isolates


In [ ]:
import os
import torch.nn as nn
from PIL import Image as PILImage

# Loss functions
# Reference: Milletari et al. (2016) V-Net (Dice Loss);
#            Lin et al. (2017) Focal Loss for class imbalance motivation.

class DiceLoss(nn.Module):
    """
    Soft Dice loss for multi-class segmentation.
    Unlike pixel-wise CE, Dice measures region-level overlap — this
    prevents the model from ignoring rare small classes (Car_Bus,
    Pedestrian) whose pixels are swamped by road/sky in CE gradients.
    """
    def __init__(self, num_classes=14, smooth=1.0, ignore_index=13):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)          # (B, C, H, W)
        B, C, H, W = probs.shape

        targets_clamped = targets.clamp(0, C - 1)
        targets_oh = F.one_hot(targets_clamped, num_classes=C)  # (B,H,W,C)
        targets_oh = targets_oh.permute(0, 3, 1, 2).float()     # (B,C,H,W)

        # Mask out ignored pixels
        valid = (targets != self.ignore_index).unsqueeze(1).float()
        probs = probs * valid
        targets_oh = targets_oh * valid

        dice_scores = []
        for c in range(C):
            if c == self.ignore_index:
                continue
            p = probs[:, c].reshape(B, -1)
            g = targets_oh[:, c].reshape(B, -1)
            inter = (p * g).sum(dim=1)
            dice = (2 * inter + self.smooth) / (p.sum(dim=1) + g.sum(dim=1) + self.smooth)
            dice_scores.append(dice)

        return 1.0 - torch.stack(dice_scores, dim=1).mean()


class CombinedLoss(nn.Module):
    """
    alpha * weighted_CE + (1-alpha) * Dice.
    CE gives stable per-pixel gradients; Dice provides region-level
    supervision that is not diluted by dominant classes.
    """
    def __init__(self, class_weights, alpha=0.5, ignore_index=13):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
        self.dice = DiceLoss(num_classes=len(class_weights), ignore_index=ignore_index)
        self.alpha = alpha

    def forward(self, logits, targets):
        return self.alpha * self.ce(logits, targets) + (1 - self.alpha) * self.dice(logits, targets)


def compute_class_weights(dataset, num_classes=14, ignore_index=13):
    """
    Inverse-frequency weights from training set pixel counts.
    Rare classes (Car_Bus=10, Pedestrian=11) get higher weights,
    penalizing the model more when it misses them.
    """
    counts = np.zeros(num_classes, dtype=np.float64)
    ldr = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)
    with torch.no_grad():
        for _, labels, _ in ldr:
            for c in range(num_classes):
                counts[c] += (labels.numpy() == c).sum()

    counts[ignore_index] = 0
    freq = counts / (counts.sum() + 1e-8)
    weights = np.zeros(num_classes, dtype=np.float64)
    valid = freq > 0
    weights[valid] = 1.0 / (freq[valid] + 1e-8)
    weights[ignore_index] = 0.0
    weights[valid] = weights[valid] / weights[valid].mean()  # normalize to mean≈1
    return torch.tensor(weights, dtype=torch.float32)


# Small-object zoom-in augmentation
# Reference: Kisantal et al. (2019) "Augmentation for Small Object
# Detection", arXiv:1902.07296 —> copy-paste approach for rare objects.
# Adapted here as crop-zoom to increase effective resolution of
# Car_Bus and Pedestrian pixels without introducing artifacts.

class SmallObjectZoomIn:
    """
    With probability p, crops a region centred on target-class pixels
    (Car_Bus=10, Pedestrian=11) and resizes to full output_size.
    Effect: small distant objects occupy more pixels → easier to learn.
    Trade-off: context around the object is reduced; may lower IoU for
    large-area classes like Road or Sky that provide spatial cues.
    """
    def __init__(self, output_size, target_class_ids=(10, 11), p=0.5, min_crop_ratio=0.35):
        self.output_size = (int(output_size), int(output_size))
        self.target_class_ids = set(target_class_ids)
        self.p = p
        self.min_crop_ratio = min_crop_ratio

    def __call__(self, image, target):
        if random.random() >= self.p:
            return image, target

        lbl_np = np.array(target, dtype=np.uint8)
        W, H = image.size  # PIL size is (W, H)

        mask = np.zeros((H, W), dtype=bool)
        for cid in self.target_class_ids:
            mask |= (lbl_np == cid)

        min_side = int(min(H, W) * self.min_crop_ratio)

        if mask.sum() == 0:
            # No target pixels — random crop as fallback
            crop_size = random.randint(min_side, min(H, W))
            top  = random.randint(0, H - crop_size)
            left = random.randint(0, W - crop_size)
        else:
            rows = np.where(np.any(mask, axis=1))[0]
            cols = np.where(np.any(mask, axis=0))[0]
            r0, r1 = rows[0], rows[-1]
            c0, c1 = cols[0], cols[-1]

            # Pad around bbox with 50% of bbox size for context
            obj_h = max(r1 - r0 + 1, 1)
            obj_w = max(c1 - c0 + 1, 1)
            pad = max(obj_h, obj_w) // 2
            crop_size = max(max(obj_h, obj_w) + 2 * pad, min_side)
            crop_size = min(crop_size, min(H, W))

            cy = (r0 + r1) // 2
            cx = (c0 + c1) // 2
            top  = max(0, min(cy - crop_size // 2, H - crop_size))
            left = max(0, min(cx - crop_size // 2, W - crop_size))

        box = (left, top, left + crop_size, top + crop_size)
        image  = image.crop(box).resize((self.output_size[1], self.output_size[0]), PILImage.BICUBIC)
        target = target.crop(box).resize((self.output_size[1], self.output_size[0]), PILImage.NEAREST)
        return image, target


In [ ]:
# Training utilities

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total, count = 0.0, 0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        total += loss.item(); count += 1
    return total / max(count, 1)


def eval_loss(model, loader, loss_fn, device):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            total += loss_fn(model(images), labels).item(); count += 1
    return total / max(count, 1)


def run_training(model, train_loader, val_loader, loss_fn, optimizer,
                 epochs, device, save_path, scheduler=None, print_every=5):
    history = {"train_loss": [], "val_loss": [], "best_val_loss": float("inf"), "best_epoch": -1}
    for epoch in range(epochs):
        tr = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        vl = eval_loss(model, val_loader, loss_fn, device)
        history["train_loss"].append(tr)
        history["val_loss"].append(vl)
        if vl < history["best_val_loss"]:
            history["best_val_loss"] = vl
            history["best_epoch"] = epoch
            torch.save(model.state_dict(), save_path)
        if scheduler is not None:
            scheduler.step(vl)
        if (epoch + 1) % print_every == 0 or epoch == 0:
            best_tag = " ← best" if epoch == history["best_epoch"] else ""
            print(f"  epoch {epoch+1:03d}/{epochs}  train={tr:.4f}  val={vl:.4f}{best_tag}")
    print(f"Done. Best val={history['best_val_loss']:.4f} at epoch {history['best_epoch']+1}")
    return history


def plot_curves(histories, labels, title="Training Curves"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    colours = ["steelblue", "darkorange", "forestgreen", "crimson"]
    for i, (hist, label) in enumerate(zip(histories, labels)):
        c = colours[i % len(colours)]
        ax1.plot(hist["train_loss"], color=c, label=label)
        ax2.plot(hist["val_loss"],   color=c, linestyle="--", label=label)
    for ax, ytitle in zip([ax1, ax2], ["Train Loss", "Val Loss"]):
        ax.set(xlabel="Epoch", ylabel="Loss", title=ytitle)
        ax.legend(); ax.grid(True)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# Setup: result directory, dataloaders, class weights
# Run this cell once before any ablation.

RESULT_DIR    = "../train_part3_results/"
EPOCHS_ABL    = 60   # ablations 0-2
EPOCHS_FINAL  = 100  # final method
LR            = 2e-4

os.makedirs(RESULT_DIR, exist_ok=True)

# Standard augmentation (shared by ablations 0-2)
std_aug = aug.Compose([
    aug.RandomHorizontalFlip(0.5),
    aug.RandomResizedCrop(IMG_SIZE[0], scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    aug.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
])

# Final augmentation: standard + zoom-in on Car_Bus/Pedestrian
final_aug = aug.Compose([
    aug.RandomHorizontalFlip(0.5),
    SmallObjectZoomIn(IMG_SIZE[0], target_class_ids=(10, 11), p=0.5),
    aug.RandomResizedCrop(IMG_SIZE[0], scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    aug.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
])

train_ds_std   = camvidLoader(DATA_ROOT, "train", is_aug=True,  aug=std_aug,   img_size=IMG_SIZE, is_pytorch_transform=True)
train_ds_final = camvidLoader(DATA_ROOT, "train", is_aug=True,  aug=final_aug, img_size=IMG_SIZE, is_pytorch_transform=True)
val_ds_c2      = camvidLoader(DATA_ROOT, "val",   is_aug=False, img_size=IMG_SIZE, is_pytorch_transform=True)

train_ld_std   = DataLoader(train_ds_std,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
train_ld_final = DataLoader(train_ds_final, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
val_ld_c2      = DataLoader(val_ds_c2,      batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

# Compute inverse-frequency class weights from training set
print("Computing class weights from training set...")
class_weights = compute_class_weights(train_ds_std, NUM_CLASSES, ignore_index=13)
class_weights_dev = class_weights.to(device)
print("Class weights:")
for idx, w in enumerate(class_weights):
    print(f"  {idx:02d} {IDX_TO_CLASS[idx]:<14}: {w:.3f}")

In [ ]:
# Ablation 0: plain CE, standard aug
# Purpose: establish the from-scratch retrained baseline for comparison.
print("=== Ablation 0: CE (standard aug) ===")
torch.manual_seed(SEED)

model_a0 = UNet(3, NUM_CLASSES, width=32, bilinear=True).to(device)
loss_a0   = nn.CrossEntropyLoss(ignore_index=13)
opt_a0    = torch.optim.Adam(model_a0.parameters(), lr=LR)

hist_a0 = run_training(
    model_a0, train_ld_std, val_ld_c2, loss_a0, opt_a0,
    epochs=EPOCHS_ABL, device=device,
    save_path=os.path.join(RESULT_DIR, "ablation0_ce.pt"),
)


In [ ]:
# Ablation 1: weighted CE, standard aug
# Isolates the effect of class weighting alone.
print("=== Ablation 1: Weighted CE (standard aug) ===")
torch.manual_seed(SEED)

model_a1 = UNet(3, NUM_CLASSES, width=32, bilinear=True).to(device)
loss_a1   = nn.CrossEntropyLoss(weight=class_weights_dev, ignore_index=13)
opt_a1    = torch.optim.Adam(model_a1.parameters(), lr=LR)

hist_a1 = run_training(
    model_a1, train_ld_std, val_ld_c2, loss_a1, opt_a1,
    epochs=EPOCHS_ABL, device=device,
    save_path=os.path.join(RESULT_DIR, "ablation1_wce.pt"),
)


In [ ]:
# Ablation 2: weighted CE + Dice, standard aug
# Isolates the additional benefit of region-level Dice supervision.
print("=== Ablation 2: Weighted CE + Dice (standard aug) ===")
torch.manual_seed(SEED)

model_a2 = UNet(3, NUM_CLASSES, width=32, bilinear=True).to(device)
loss_a2   = CombinedLoss(class_weights_dev, alpha=0.5, ignore_index=13)
opt_a2    = torch.optim.Adam(model_a2.parameters(), lr=LR)

hist_a2 = run_training(
    model_a2, train_ld_std, val_ld_c2, loss_a2, opt_a2,
    epochs=EPOCHS_ABL, device=device,
    save_path=os.path.join(RESULT_DIR, "ablation2_wce_dice.pt"),
)


In [ ]:
# Final method: weighted CE + Dice + zoom-in aug
# Full design: combines all improvements. Trained longer (60 epochs).
print("=== Final: Weighted CE + Dice + Zoom-in aug (60 epochs) ===")
torch.manual_seed(SEED)

model_final = UNet(3, NUM_CLASSES, width=32, bilinear=True).to(device)
loss_final  = CombinedLoss(class_weights_dev, alpha=0.5, ignore_index=13)
opt_final   = torch.optim.Adam(model_final.parameters(), lr=LR)
sched_final = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_final, mode="min", patience=8, factor=0.5)

hist_final = run_training(
    model_final, train_ld_final, val_ld_c2, loss_final, opt_final,
    epochs=EPOCHS_FINAL, device=device,
    save_path=os.path.join(RESULT_DIR, "final_model.pt"),
    scheduler=sched_final,
)

In [ ]:
# Figure B1 equivalent for Part C: training curves
plot_curves(
    [hist_a0, hist_a1, hist_a2, hist_final],
    ["Abl-0: CE", "Abl-1: Weighted CE", "Abl-2: W-CE+Dice", "Final: W-CE+Dice+ZoomIn"],
    title="C2 Ablation Training Curves (Val Split Used for Checkpoint Selection)",
)


## Part C3 — compare ablations on **validation**, then test the final selected method once

Your report should include:
- baseline vs improved comparison
- at least two ablations
- object-level results for your assigned pair
- qualitative examples
- trade-off discussion


In [ ]:
# Val comparison table for all ablations

def load_and_eval(model_path, loader, device, num_classes=14):
    """Load saved state_dict and evaluate on loader."""
    model = UNet(3, num_classes, width=32, bilinear=True).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    return evaluate_model_on_loader(model, loader, device, num_classes)


ablation_configs = [
    ("Abl-0: CE (baseline)",     os.path.join(RESULT_DIR, "ablation0_ce.pt")),
    ("Abl-1: Weighted CE",       os.path.join(RESULT_DIR, "ablation1_wce.pt")),
    ("Abl-2: W-CE + Dice",       os.path.join(RESULT_DIR, "ablation2_wce_dice.pt")),
    ("Final: W-CE+Dice+ZoomIn",  os.path.join(RESULT_DIR, "final_model.pt")),
]

abl_results = {}
for label, path in ablation_configs:
    print(f"Evaluating: {label} ...")
    abl_results[label] = load_and_eval(path, val_ld_c2, device, NUM_CLASSES)

car_idx = CLASS_LABELS_DICT["Car_Bus"]
ped_idx = CLASS_LABELS_DICT["Pedestrian"]

print("\n" + "=" * 75)
print(f"{'Method':<30} {'GlobalAcc':>10} {'mIoU':>8} {'Car_Bus':>9} {'Pedestrian':>11}")
print("=" * 75)
for label, res in abl_results.items():
    print(
        f"{label:<30}"
        f" {res['overall_global_accuracy']:>10.4f}"
        f" {res['mean_iou']:>8.4f}"
        f" {res['per_class_iou'][car_idx]:>9.4f}"
        f" {res['per_class_iou'][ped_idx]:>11.4f}"
    )
print("=" * 75)
print("\nNote: all results from validation split.")


In [ ]:
# Compute_object_level_stats_percentile for all ablations
# Shows whether small-object precision/recall/F1 improves step by step.

for label, res in abl_results.items():
    gt_arr   = np.array(res["gt_masks"])
    pred_arr = np.array(res["pred_masks"])
    print(f"\n{'='*70}")
    print(f"Object-level stats — {label}")
    for cls_name in FOCUS_CLASSES:
        print(f"\n  >> {cls_name}")
        compute_object_level_stats_percentile(
            gt_masks=gt_arr,
            pred_masks=pred_arr,
            class_label=cls_name,
            iou_threshold=0.2,
            size_percentile=50,
        )


In [ ]:
# Final model on test split, evaluated ONCE only
# Do not use these results to guide further changes.

test_dataset = camvidLoader(
    DATA_ROOT, "test", is_aug=False, img_size=IMG_SIZE, is_pytorch_transform=True
)
test_loader_c3 = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, drop_last=False,
)
print(f"Test images: {len(test_dataset)}")

final_model_test = UNet(3, NUM_CLASSES, width=32, bilinear=True).to(device)
final_model_test.load_state_dict(
    torch.load(os.path.join(RESULT_DIR, "final_model.pt"), map_location=device)
)

test_res = evaluate_model_on_loader(final_model_test, test_loader_c3, device, NUM_CLASSES)

print("\n=== FINAL MODEL | SUNNY TEST (hold-out, one-time evaluation) ===")
print(f"Global Accuracy : {test_res['overall_global_accuracy']:.4f}")
print(f"Mean IoU        : {test_res['mean_iou']:.4f}")
print("\nPer-Class IoU:")
for idx, iou in enumerate(test_res["per_class_iou"]):
    name = IDX_TO_CLASS[idx]
    print(f"  {idx:02d} {name:<14}: {'nan' if np.isnan(iou) else f'{iou:.4f}'}")

# Object-level stats on test for focus classes
print("\n--- Object-level stats on TEST (final model) ---")
gt_test   = np.array(test_res["gt_masks"])
pred_test = np.array(test_res["pred_masks"])
for cls_name in FOCUS_CLASSES:
    print(f"\n  >> {cls_name}")
    compute_object_level_stats_percentile(
        gt_masks=gt_test,
        pred_masks=pred_test,
        class_label=cls_name,
        iou_threshold=0.2,
        size_percentile=50,
    )


In [ ]:
# Side-by-side qualitative comparison
# For each focus class, find 5 images with the largest IoU gain
# between Abl-0 (baseline) and the final model.

baseline_res = abl_results["Abl-0: CE (baseline)"]
final_res    = abl_results["Final: W-CE+Dice+ZoomIn"]


def class_iou_single(gt, pred, class_id):
    tp = np.logical_and(gt == class_id, pred == class_id).sum()
    fp = np.logical_and(gt != class_id, pred == class_id).sum()
    fn = np.logical_and(gt == class_id, pred != class_id).sum()
    d = tp + fp + fn
    return tp / d if d > 0 else 0.0


def find_improved_images(baseline_res, final_res, class_id, top_n=5, min_gt_pixels=30):
    """Images where final model most improves over baseline for class_id."""
    records = []
    for i, gt in enumerate(baseline_res["gt_masks"]):
        gt = np.array(gt)
        if (gt == class_id).sum() < min_gt_pixels:
            continue
        iou_b = class_iou_single(gt, np.array(baseline_res["pred_masks"][i]), class_id)
        iou_f = class_iou_single(gt, np.array(final_res["pred_masks"][i]),    class_id)
        records.append((i, iou_b, iou_f, iou_f - iou_b))
    records.sort(key=lambda x: -x[3])  # largest improvement first
    return records[:top_n]


def show_comparison(records, baseline_res, final_res, class_name):
    """4-column grid: original | GT | baseline pred | final pred."""
    n = len(records)
    fig, axes = plt.subplots(n, 4, figsize=(20, 4.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original", "Ground Truth", "Baseline Pred", "Final Pred"]):
        axes[0, col].set_title(title, fontsize=10, fontweight="bold")

    for row, (img_idx, iou_b, iou_f, delta) in enumerate(records):
        img = plt.imread(baseline_res["image_paths"][img_idx])
        gt  = baseline_res["gt_masks"][img_idx]
        pb  = baseline_res["pred_masks"][img_idx]
        pf  = final_res["pred_masks"][img_idx]

        axes[row, 0].imshow(img)
        axes[row, 0].set_ylabel(
            f"Img {img_idx}\n{class_name}\nBase={iou_b:.3f}\nFinal={iou_f:.3f}\n(+{delta:.3f})",
            fontsize=8, rotation=0, labelpad=115, va="center",
        )
        axes[row, 1].imshow(from_label_to_rgb(gt))
        axes[row, 2].imshow(from_label_to_rgb(pb))
        axes[row, 3].imshow(from_label_to_rgb(pf))
        for ax in axes[row]:
            ax.axis("off")

    fig.suptitle(
        f"C3 Qualitative Comparison | {class_name} (Val Split, largest IoU gains)",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


for cls_name, cls_id in FOCUS_CLASS_IDS.items():
    records = find_improved_images(baseline_res, final_res, class_id=cls_id, top_n=5)
    show_comparison(records, baseline_res, final_res, class_name=cls_name)
